# Check the evaluation boundary

**Optional code walkthrough for AI 101 (Module 10).** Not graded. Quiz scores do not depend on it.

**Objective.** Reproduce the five-fold counts from Lesson 10.1. Then check two kinds of leakage with very small examples: the same machine in both training and test data, and a mean fitted on data that includes the held-out value.

**Supports.** Lesson 10.1 (section “Cross-validation stays inside development data”) and Lesson 10.3 (sections “Leakage violates an information boundary” and “Preprocessing can leak without using target labels”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

Steps 1 and 2 use the numbers of Lesson 10.1: 100 independent examples in five folds. Step 3 uses six invented rows from three machines. Step 4 uses five invented numbers.

All data are synthetic and dimensionless. They are invented for teaching and are not measurements from real equipment.

```
Step 3: six rows from three machines
row 1: M1    row 2: M1
row 3: M2    row 4: M2
row 5: M3    row 6: M3

Step 4: training values 10, 12, 14, 16 and one held-out value 40
```


## Step 1. Count the examples in one fold

Lesson 10.1 divides 100 independent examples into five equal folds. In each iteration one fold is held out for validation. The other four folds are used for training.

In [ ]:
# Step 1: five-fold counts from Lesson 10.1
n_examples = 100
n_folds = 5
fold_size = n_examples // n_folds        # examples in one fold
n_train = n_examples - fold_size         # examples left for training
print("Examples per fold:", fold_size)
print("In one iteration:", n_train, "for training and", fold_size, "for validation")

**Expected output**

```
Examples per fold: 20
In one iteration: 80 for training and 20 for validation
```

This matches Lesson 10.1: 20 examples per fold, 80 for training and 20 for validation. These are counts of examples, not physical quantities.

## Step 2. List the five validation folds

Now you list which examples each fold holds out. The examples are numbered 0 to 99. Fold 1 holds out examples 0 to 19, fold 2 holds out 20 to 39, and so on. Every example is held out exactly once.

In [ ]:
# Step 2: the validation examples of each fold
for fold in range(n_folds):
    first = fold * fold_size
    last = first + fold_size - 1
    print(f"Fold {fold + 1}: validation examples {first} to {last}; {n_train} examples left for training")

**Expected output**

```
Fold 1: validation examples 0 to 19; 80 examples left for training
Fold 2: validation examples 20 to 39; 80 examples left for training
Fold 3: validation examples 40 to 59; 80 examples left for training
Fold 4: validation examples 60 to 79; 80 examples left for training
Fold 5: validation examples 80 to 99; 80 examples left for training
```

The five validation parts do not overlap and together cover all 100 examples. Cross-validation stays inside the development data. It does not replace a separate final test set (Lesson 10.1). This simple numbering is only correct when the examples are independent, as in the lesson.

## Step 3. Check whether a machine appears on both sides of a split

Lesson 10.1 says that predicting a machine absent from development is a different question from predicting later rows of a known machine. Here the function `shared_machines` lists the machines that appear in both the training rows and the test rows. An empty list means that no machine appears on both sides.

In [ ]:
# Step 3: which machines appear in both training and test rows?
rows = [("row 1", "M1"), ("row 2", "M1"),
        ("row 3", "M2"), ("row 4", "M2"),
        ("row 5", "M3"), ("row 6", "M3")]

def shared_machines(test_rows):
    train_machines = []
    test_machines = []
    for row, machine in rows:
        if row in test_rows:
            test_machines.append(machine)
        else:
            train_machines.append(machine)
    shared = []
    for machine in test_machines:
        if machine in train_machines and machine not in shared:
            shared.append(machine)
    return shared

print("Mixed-rows split, machines in both sets:", shared_machines(["row 2", "row 4", "row 6"]))
print("Machine hold-out split, machines in both sets:", shared_machines(["row 5", "row 6"]))

**Expected output**

```
Mixed-rows split, machines in both sets: ['M1', 'M2', 'M3']
Machine hold-out split, machines in both sets: []
```

In the mixed-rows split every test machine also appears in training. The model could recognise those machines. In the hold-out split, machine M3 is completely absent from training. Only this second split can show how the procedure behaves on a machine it has not seen. Use it when the intended use is a new machine.

## Step 4. See a preprocessing leak

Lesson 10.3 says that fitting a scaler on all data lets held-out examples influence the training procedure. Here the “fitted value” is a mean. The correct way fits the mean on the training values only. The leaky way includes the held-out value. Then each value is centred by subtracting the mean.

In [ ]:
# Step 4: a mean fitted with and without the held-out value
train_values = [10, 12, 14, 16]
held_out = [40]

mean_train = sum(train_values) / len(train_values)
mean_all = (sum(train_values) + sum(held_out)) / (len(train_values) + len(held_out))

print(f"Mean fitted on training values only: {mean_train:.1f}")
print(f"Mean fitted on all values: {mean_all:.1f}")
print(f"Held-out value after centring (correct): {held_out[0] - mean_train:.1f}")
print(f"Held-out value after centring (leaky): {held_out[0] - mean_all:.1f}")

**Expected output**

```
Mean fitted on training values only: 13.0
Mean fitted on all values: 18.4
Held-out value after centring (correct): 27.0
Held-out value after centring (leaky): 21.6
```

The leaky mean (18.4) moved toward the held-out value 40. The held-out example made itself look less unusual. The correct procedure fits the mean on training values only and then applies that stored value to the held-out example. The same idea applies to a scaler or an imputer inside every cross-validation fold.

## Try it

Change a value and observe. In Step 4, change `held_out = [40]` to `held_out = [60]`. Then run the cell again.

**Expected result**

```
Mean fitted on training values only: 13.0
Mean fitted on all values: 22.4
Held-out value after centring (correct): 47.0
Held-out value after centring (leaky): 37.6
```

The correct mean stays at 13.0, because it never sees the held-out value. The leaky mean moves from 18.4 to 22.4, so the leak grows when the held-out value is further from the training values.

## Limits

- The data are tiny, synthetic and dimensionless. They show how to count and check. They do not show how large a real leak would be.
- Step 2 numbers the folds in order. This is only correct for independent examples. For time series or grouped data, use the splits taught in Lessons 10.1 and 10.3.
- Step 4 uses a mean as a very simple fitted value. A real scaler or imputer is checked in the same way: which examples contributed to its fitted values?
- The code checks a boundary. It does not fit or evaluate a model.

Lab 2 (Module 7) and Lab 3 (Module 8) use a train and test split. This walkthrough shows how to check the boundary behind such a split.